# Pseudo-Label: Confidence-threshold-only control (mask unconfident pseudo-labels, no lock)
### Control Ablation: Masking Unconfident Pseudo-Labels Without Sticky Lock

This notebook evaluates standard pseudo-labeling with a confidence threshold mask ($p_{\max} \ge 0.9$ and others $< 0.3$) versus standard `pl` under $n=30$ labeled examples. Unconfident samples contribute zero loss, but pseudo-labels remain dynamic and are never locked across epochs.
- **Idea**: Test whether masking unconfident predictions alone prevents confirmation bias and label noise.
- **Modes Compared**: `pl` vs `thr`.

In [3]:
# Standalone Colab snippet:
# !git clone https://github.com/marcsanchezsala/semi-supervised-mnist.git
# %cd semi-supervised-mnist/04_threshold_only
import os
import sys
import json
import numpy as np
import torch

if os.path.isdir("04_threshold_only"):
    os.chdir("04_threshold_only")
if "../common" not in sys.path:
    sys.path.append("../common")

from pl import load_mnist, make_splits, train, test_error, device

print(f"Using device: {device}")

Using device: cuda


In [4]:
X_train, y_train, X_test, y_test = load_mnist()
print(f"Loaded MNIST: X_train {X_train.shape}, X_test {X_test.shape}")

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.77MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 136kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.08MB/s]


Loaded MNIST: X_train torch.Size([60000, 784]), X_test torch.Size([10000, 784])


In [5]:
N_LABELED = 30
N_SEEDS = 3  # shortcut: 3 paired seeds (same split + init per mode), upgrade to 10 for tighter CI
MODES = ["pl", "thr"]
results = {m: [] for m in MODES}

for s in range(N_SEEDS):
    seed = 42 + s
    X_l, y_l, X_u, y_u, X_val, y_val = make_splits(X_train, y_train, n_labeled=N_LABELED, seed=seed)

    for mode in MODES:
        torch.manual_seed(seed)
        model, info = train(
            X_l, y_l, X_u, X_val, y_val, X_test, y_test,
            mode=mode, y_u=y_u, verbose=False
        )
        info["seed"] = seed
        info["mode"] = mode
        info["test_val_sel"] = test_error(model, X_test, y_test)
        results[mode].append(info)

        print(
            f"Seed {seed} | Mode {mode:7s} | "
            f"Val-sel Test Err: {info['test_val_sel']:5.2f}% | "
            f"Last Test Err: {info['test_last']:5.2f}% | "
            f"Cov: {info['coverage']:5.2f}% | "
            f"Prec: {info['precision']:5.2f}% | "
            f"Time: {info['seconds']:5.1f}s | "
            f"Peak: {info['peak_mb']:6.1f} MB",
            flush=True,
        )

        with open("results.json", "w") as f:
            json.dump(results, f, indent=2, default=float)

Seed 42 | Mode pl      | Val-sel Test Err: 27.16% | Last Test Err: 27.84% | Cov:   nan% | Prec:   nan% | Time: 119.0s | Peak: 1051.3 MB
Seed 42 | Mode thr     | Val-sel Test Err: 22.57% | Last Test Err: 22.75% | Cov: 97.44% | Prec: 77.08% | Time: 110.4s | Peak: 1071.7 MB
Seed 43 | Mode pl      | Val-sel Test Err: 39.09% | Last Test Err: 42.22% | Cov:   nan% | Prec:   nan% | Time: 104.6s | Peak: 1066.5 MB
Seed 43 | Mode thr     | Val-sel Test Err: 33.44% | Last Test Err: 33.42% | Cov: 97.58% | Prec: 65.96% | Time: 110.2s | Peak: 1071.1 MB
Seed 44 | Mode pl      | Val-sel Test Err: 36.82% | Last Test Err: 37.06% | Cov:   nan% | Prec:   nan% | Time: 104.7s | Peak: 1067.3 MB
Seed 44 | Mode thr     | Val-sel Test Err: 22.98% | Last Test Err: 22.93% | Cov: 97.23% | Prec: 77.26% | Time: 110.2s | Peak: 1071.8 MB


In [6]:
print("=" * 90)
print(f"{'Mode':<10} | {'Val-sel Test Err':>18} | {'Last Test Err':>16} | {'Coverage':>10} | {'Precision':>10} | {'Time (s)':>10} | {'Peak (MB)':>10}")
print("-" * 90)

for mode in MODES:
    runs = results[mode]
    val_sel_errs = [r["test_val_sel"] for r in runs]
    last_errs = [r["test_last"] for r in runs]
    covs = [r["coverage"] for r in runs]
    precs = [r["precision"] for r in runs]
    secs = [r["seconds"] for r in runs]
    peaks = [r["peak_mb"] for r in runs]

    val_str = f"{np.mean(val_sel_errs):5.2f} ± {np.std(val_sel_errs):4.2f}%"
    last_str = f"{np.mean(last_errs):5.2f} ± {np.std(last_errs):4.2f}%"
    cov_str = f"{np.nanmean(covs):5.2f}%" if not np.isnan(covs).all() else "N/A"
    prec_str = f"{np.nanmean(precs):5.2f}%" if not np.isnan(precs).all() else "N/A"
    sec_str = f"{np.mean(secs):6.1f}s"
    peak_str = f"{np.mean(peaks):6.1f} MB"

    print(f"{mode:<10} | {val_str:>18} | {last_str:>16} | {cov_str:>10} | {prec_str:>10} | {sec_str:>10} | {peak_str:>10}")

print("=" * 90)

if "pl" in MODES and len(MODES) > 1:
    other_modes = [m for m in MODES if m != "pl"]
    pl_runs = results["pl"]
    for other in other_modes:
        other_runs = results[other]
        diff_val = [other_runs[i]["test_val_sel"] - pl_runs[i]["test_val_sel"] for i in range(len(pl_runs))]
        diff_last = [other_runs[i]["test_last"] - pl_runs[i]["test_last"] for i in range(len(pl_runs))]

        mean_d_val, std_d_val = np.mean(diff_val), np.std(diff_val)
        mean_d_last, std_d_last = np.mean(diff_last), np.std(diff_last)

        print(f"\nPaired per-seed difference ({other} - pl):")
        print(f"  Val-selected test err diff: {mean_d_val:+5.2f} ± {std_d_val:4.2f}%")
        print(f"  Last-epoch test err diff:   {mean_d_last:+5.2f} ± {std_d_last:4.2f}%")

        if mean_d_val < -std_d_val:
            verdict = f"improves over PL by > 1 paired std (diff={mean_d_val:+.2f}%, std={std_d_val:.2f}%)"
        else:
            verdict = f"does NOT improve over PL by > 1 paired std (diff={mean_d_val:+.2f}%, std={std_d_val:.2f}%)"
        print(f"Verdict: {other} {verdict}")

Mode       |   Val-sel Test Err |    Last Test Err |   Coverage |  Precision |   Time (s) |  Peak (MB)
------------------------------------------------------------------------------------------
pl         |      34.36 ± 5.17% |    35.71 ± 5.95% |        N/A |        N/A |     109.4s |  1061.7 MB
thr        |      26.33 ± 5.03% |    26.37 ± 4.99% |     97.42% |     73.43% |     110.2s |  1071.5 MB

Paired per-seed difference (thr - pl):
  Val-selected test err diff: -8.03 ± 4.13%
  Last-epoch test err diff:   -9.34 ± 3.71%
Verdict: thr improves over PL by > 1 paired std (diff=-8.03%, std=4.13%)
